# DATA_PREPARE — pipeline dữ liệu của đồ án

Chạy **một lần**, khoảng 20 phút trên Colab, **không cần GPU**.

```
Zenodo 5.7 GB
   -> giai nen 13 GB CSV  (external/mobivital/dataset/mobivital/tripod/, ban duy nhat)
   -> data/processed/by_user/*.npz     scripts/make_npz.py
   -> data/processed/windows/          scripts/make_windows.py
   -> ~2.7 GB len Google Drive
```

Trong đây có chạy `prep_breath_final.py` của MobiVital để sinh `data_final/*.npy` —
không dùng về sau, chỉ để **đối chiếu**: chứng minh `by_user/*.npz` chứa đúng từng
byte những gì code gốc đọc ra (mục 8).

## 0. Môi trường — đĩa trống, GPU

Cần ít nhất **25 GB trống**: zip 5.7 GB + CSV 13 GB + npz 3 GB.

In [1]:
import os
import subprocess


def sh(command):
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return (done.stdout + done.stderr).strip()


print(sh("df -h /content | tail -1"))
print("nvidia-smi:", sh("nvidia-smi --query-gpu=name --format=csv,noheader") or "no GPU")


overlay         236G   48G  189G  21% /
nvidia-smi: Tesla T4


## 1. Google Drive

Nơi cất dữ liệu đã xử lý. Cần bấm popup OAuth — nếu bỏ qua thì mục 11 không cất được, phần còn lại vẫn chạy.

In [18]:
# Drive mount cần bấm popup OAuth — chạy tay ô này. Ở đây tạm bỏ qua.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/mobivital"
    os.makedirs(DRIVE, exist_ok=True)
    print("cất kết quả vào", DRIVE)
except Exception as e:
    DRIVE = None
    print("bỏ qua Drive:", e)


Mounted at /content/drive
cất kết quả vào /content/drive/MyDrive/mobivital


## 2. Code

Clone repo đồ án và upstream MobiVital (`4319731`). `external/mobivital/` bị `.gitignore` chặn nên clone riêng mỗi phiên.

In [4]:
REPO = "/content/UWB_RADAR"

if os.path.exists(REPO + "/.git"):
    os.chdir(REPO)
    print(sh("git pull -q origin main && echo 'code đã mới nhất'"))
else:
    os.chdir("/content")
    sh("rm -rf " + REPO)
    print(sh("git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git " + REPO))
    print(sh("git clone -q https://github.com/nesl/mobivital-public.git " + REPO + "/external/mobivital"))
    print(sh("pip install -q einops"))

os.chdir(REPO)
print("đứng ở         :", os.getcwd())
print("commit đồ án :", sh("git rev-parse --short HEAD"))
print("commit MobiVital:", sh("git -C external/mobivital rev-parse --short HEAD"))





đứng ở         : /content/UWB_RADAR
commit của mình : a40b765
commit MobiVital: 4319731


## 3. Tải dataset từ Zenodo

`aria2c` 16 luồng — 2 phút. `wget` một luồng mất 2.3 giờ vì Zenodo bóp băng thông mỗi kết nối.

In [5]:
ZIP = "/content/tripod.zip"
ZIP_SIZE = 5700705593      # byte, từ Zenodo API

if os.path.exists(ZIP) and os.path.getsize(ZIP) == ZIP_SIZE:
    print("đã có, bỏ qua bước tải")
else:
    sh("apt-get install -qq -y aria2")
    out = sh("aria2c -x16 -s16 -k5M --summary-interval=0 --console-log-level=error "
             "-d /content -o tripod.zip "
             "https://zenodo.org/api/records/15022885/files/tripod.zip/content")
    for line in out.split("\n"):
        if "OK" in line or "ERR" in line:
            print(line)

size = os.path.getsize(ZIP)
print("%.2f GB   đúng kích thước Zenodo: %s" % (size / 1e9, size == ZIP_SIZE))


ed0f07|OK  |    25MiB/s|/content/tripod.zip
(OK):download completed.
5.70 GB   đúng kích thước Zenodo: True


## 4. Giải nén — một bản CSV duy nhất

Thẳng vào `external/mobivital/dataset/mobivital/tripod/`, đúng đường dẫn `prep_breath_final.py` dòng 18 đòi. `data/` chỉ chứa thứ pipeline của đồ án sinh ra. Zip đã có sẵn thư mục `tripod/` nên giải nén vào `.../mobivital/`.

In [6]:
CSV_DIR = "external/mobivital/dataset/mobivital/tripod"

os.makedirs("external/mobivital/dataset/mobivital", exist_ok=True)
sh("unzip -q -o " + ZIP + " -d external/mobivital/dataset/mobivital/")

print("số file CSV:", sh("ls " + CSV_DIR + " | wc -l"))
print("dung lượng :", sh("du -sh " + CSV_DIR + " | cut -f1"))
print("file mẫu   :", sorted(os.listdir(CSV_DIR))[0])


số file CSV: 1874
dung lượng : 13G
file mẫu   : 231012_userI_tripod_01_0.csv


## 5. Dọn chỗ cho pipeline gốc

`setup_dataset.py` thêm hai thứ MobiVital không có: thư mục lối tắt **vá 52 tên file lỗi thời** (bảng kết quả tác giả commit trước khi Zenodo đổi tên, mốc tháng 10 → tháng 12), và `.git/info/exclude` để giấu dữ liệu khỏi git của MobiVital. Không đụng dòng code nào.

In [7]:
print(sh("python scripts/mobivital/setup_dataset.py"))


1874 file CSV trong external/mobivital/dataset/mobivital/tripod

1. sao lưu bảng của họ -> results/TN0a.txt (537 dòng)
2. external/mobivital/dataset/mobivital/tripod_old_names -> vá 52 tên cũ, 1926 lối tắt
3. thêm dataset/ data_final/ scores*.csv vào .git/info/exclude

KIỂM TRA
   tên trong bảng của họ : 537
   mở không được         : 0  <- phải là 0
   thư mục CSV thật      : 1874  <- phải là 1874

Xong. Giờ chạy được đúng lệnh trong README của MobiVital:
   cd external/mobivital
   python dataset_preparation/prep_breath_final.py
   python -m inference.evaluate -m tripod_mobivital_pre_invert_0.9.txt -d ./dataset/mobivital/tripod_old_names
   python -m inference.mobivital_gen
   python -m training.autoreg_training --model_name lstm_retrained


## 6. → `data_final/*.npy` — chạy chính script của MobiVital

Lệnh đầu tiên trong README của MobiVital, nguyên bản. Lọc `ABCDEFKL` → `training_...npy` (1289 buổi ghi), `GHIJ` → `testing_...npy` (537). `2>/dev/null` bỏ tiến trình `tqdm`.

In [9]:
NPY = "external/mobivital/data_final"

if os.path.exists(NPY + "/training_breath_tripod_data.npy"):
    print("data_final đã có, bỏ qua")
else:
    # tqdm ghi tiến trình bằng \r ra stderr — bỏ stderr đi, chỉ giữ stdout
    sh("cd external/mobivital && python dataset_preparation/prep_breath_final.py 2>/dev/null")

print(sh("ls -la " + NPY))
print()
print("KIỂM TRA không sửa gì trong repo MobiVital:")
print(sh("git -C external/mobivital status --short") or "  git status trống")


data_final đã có, bỏ qua
total 2578532
drwxr-xr-x  2 root root       4096 Sep  3 14:47 .
drwxr-xr-x 13 root root       4096 Sep  3 14:36 ..
-rw-r--r--  1 root root  776502256 Sep  3 14:47 testing_breath_tripod_data.npy
-rw-r--r--  1 root root 1863894256 Sep  3 14:44 training_breath_tripod_data.npy

KIỂM TRA không sửa gì trong repo MobiVital:
  git status trống


## 7. → `by_user/*.npz` — pipeline của đồ án

`make_npz.py` đọc đúng bộ CSV đó, gom theo từng người, lọc bằng tên file. Mỗi `.npz`: `uwb` (n, 1500, 120) complex64, `gt` (n, 1500) float32 chuẩn hoá [-1, 1], `files` tên CSV.

In [10]:
print(sh("python scripts/make_npz.py"))


tìm thấy 1874 file CSV
user A -> data/processed/by_user/A.npz | uwb (224, 1500, 120) | gt (224, 1500) | files (224,)
user B -> data/processed/by_user/B.npz | uwb (156, 1500, 120) | gt (156, 1500) | files (156,)
user C -> data/processed/by_user/C.npz | uwb (211, 1500, 120) | gt (211, 1500) | files (211,)
user D -> data/processed/by_user/D.npz | uwb (206, 1500, 120) | gt (206, 1500) | files (206,)
user E -> data/processed/by_user/E.npz | uwb (126, 1500, 120) | gt (126, 1500) | files (126,)
user F -> data/processed/by_user/F.npz | uwb (102, 1500, 120) | gt (102, 1500) | files (102,)
user G -> data/processed/by_user/G.npz | uwb (134, 1500, 120) | gt (134, 1500) | files (134,)
user H -> data/processed/by_user/H.npz | uwb (138, 1500, 120) | gt (138, 1500) | files (138,)
user I -> data/processed/by_user/I.npz | uwb (145, 1500, 120) | gt (145, 1500) | files (145,)
user J -> data/processed/by_user/J.npz | uwb (120, 1500, 120) | gt (120, 1500) | files (120,)
user K -> data/processed/by_user/K.np

## 8. Đối chiếu — `by_user` == `data_final`, từng byte

`check_data.py` ghép cặp từng buổi ghi bằng chữ ký md5 của `gt` (hai bên xếp thứ tự khác nhau) rồi so cả `gt` lẫn `uwb`. Cùng CSV, cùng công thức, cùng `float32` → phải giống tuyệt đối. Phải in `1289/1289` + `537/537`. Không khớp thì script tự dừng.

In [11]:
print(sh("python scripts/check_data.py"))


Đối chiếu by_user/*.npz  với  data_final/*.npy

A B C D E F K L
------------------------------------------------------------------
    external/mobivital/data_final/training_breath_tripod_data.npy
    uwb (1289, 1500, 120) complex64 | gt (1289, 1500) float32
    của mình   1289 buổi ghi
    MobiVital  1289 buổi ghi
    khớp TỪNG BYTE 1289/1289

G H I J
------------------------------------------------------------------
    external/mobivital/data_final/testing_breath_tripod_data.npy
    uwb (537, 1500, 120) complex64 | gt (537, 1500) float32
    của mình   537 buổi ghi
    MobiVital  537 buổi ghi
    khớp TỪNG BYTE 537/537

ABCDEFKL  1289/1289 buổi ghi khớp TỪNG BYTE   = training_breath_tripod_data.npy
GHIJ       537/537  buổi ghi khớp TỪNG BYTE   = testing_breath_tripod_data.npy
Từ đây mọi thí nghiệm chỉ đọc by_user/*.npz.


## 9. → `windows/` — cắt sẵn cửa sổ để train

`make_windows.py` gọi `generate_dataset` của MobiVital. `dev_cv/` cắt riêng từng người (`run_cv.py` ghép fold), `final_train/` cắt gộp 8 người đọc thẳng `data_final` (`run_final_test.py`). Cửa sổ **chỉ để train** — chấm điểm đọc buổi ghi thô, vì bước cắt này lọc sóng có nhìn nhịp thở thật (`corr > 0.9`).

In [12]:
print(sh("python scripts/make_windows.py"))
print()
print(sh("du -sh data/processed/windows/dev_cv data/processed/windows/final_train"))


PHẦN 1 — pipeline DEV, cắt riêng từng người
----------------------------------------------------------------------
ngưỡng 0.9
    data/processed/windows/dev_cv/A_corr0.9_h200_f25.npz | X (42640, 200) | 37 MB
      mất 2 giây
    data/processed/windows/dev_cv/B_corr0.9_h200_f25.npz | X (39104, 200) | 34 MB
      mất 1 giây
    data/processed/windows/dev_cv/C_corr0.9_h200_f25.npz | X (47996, 200) | 41 MB
      mất 2 giây
    data/processed/windows/dev_cv/D_corr0.9_h200_f25.npz | X (53300, 200) | 46 MB
      mất 2 giây
    data/processed/windows/dev_cv/E_corr0.9_h200_f25.npz | X (25792, 200) | 22 MB
      mất 1 giây
    data/processed/windows/dev_cv/F_corr0.9_h200_f25.npz | X (9256, 200) | 8 MB
      mất 1 giây
    data/processed/windows/dev_cv/K_corr0.9_h200_f25.npz | X (50804, 200) | 44 MB
      mất 2 giây
    data/processed/windows/dev_cv/L_corr0.9_h200_f25.npz | X (23816, 200) | 20 MB
      mất 1 giây

PHẦN 2 — pipeline GỐC, cắt gộp 8 người
--------------------------------------------

## 10. Băm nội dung — `data/checksums.txt`

Băm **nội dung mảng** 12 file `by_user/*.npz` (không băm vỏ ZIP — ZIP nhúng thời điểm ghi). Chỉ băm `by_user` vì nó tính bằng `+ - x :`, khớp TỪNG SỐ giữa các máy. `windows` dùng `np.angle/unwrap/corrcoef` — lệch chữ số cuối ~2e-8 tuỳ thư viện toán từng máy, nhỏ hơn sai số `float32`, không băm vào.


In [ ]:
print(sh("python scripts/checksums.py"))
print()
print("SO VỚI data/checksums.txt đã commit:")
print(sh("git --no-pager diff data/checksums.txt")
      or "  khớp từng dòng — by_user trên Colab == by_user đã commit từ máy")


## 11. Cất lên Drive

`windows.tar.gz` (~100 MB) + `by_user.tar` (~2.6 GB). Không đưa lên: CSV thô 13 GB (tải lại Zenodo 2 phút), `data_final/*.npy` (sinh lại bằng script MobiVital). Cần mount Drive ở mục 1.

In [20]:
if DRIVE is None:
    print("Chưa mount Drive. Chạy ô 2 (bấm popup OAuth) rồi chạy lại ô này.")
else:
    sh("tar -czf " + DRIVE + "/windows.tar.gz -C data/processed windows")
    sh("tar -cf  " + DRIVE + "/by_user.tar    -C data/processed by_user")
    print(sh("ls -la " + DRIVE))
    print()
    print("Drive đang dùng:", sh("du -sh " + DRIVE + " | cut -f1"))


total 2734650
-rw------- 1 root root 2640629760 Sep  3 15:30 by_user.tar
-rw------- 1 root root   53197934 Sep  3 08:25 windows_dev_cv.tar.gz
-rw------- 1 root root  106452993 Sep  3 15:29 windows.tar.gz

Drive đang dùng: 2.7G


## Xong

`by_user/*.npz` + `windows/` sẵn sàng. Mọi thí nghiệm sau giải nén từ Drive (2 phút) rồi chạy `scripts/run_cv.py` / `scripts/run_final_test.py`. TN0 tự dựng lại toàn bộ từ đầu để đứng độc lập.

In [16]:
print("=== DATA_PREPARE đã xong ===")
print(sh("du -sh data/processed/by_user data/processed/windows/dev_cv data/processed/windows/final_train"))
print()
print(sh("ls data/processed/by_user | tr '\n' ' '"))
print(sh("ls data/processed/windows/dev_cv | tr '\n' ' '"))
print(sh("ls data/processed/windows/final_train"))


=== DATA_PREPARE đã xong ===
2.5G	data/processed/by_user
252M	data/processed/windows/dev_cv
252M	data/processed/windows/final_train

A.npz B.npz C.npz D.npz E.npz F.npz G.npz H.npz I.npz J.npz K.npz L.npz
A_corr0.9_h200_f25.npz B_corr0.9_h200_f25.npz C_corr0.9_h200_f25.npz D_corr0.9_h200_f25.npz E_corr0.9_h200_f25.npz F_corr0.9_h200_f25.npz K_corr0.9_h200_f25.npz L_corr0.9_h200_f25.npz
train_corr0.9_h200_f25.npz
